In [108]:
pip install rugbypy

Note: you may need to restart the kernel to use updated packages.


In [109]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [110]:
from rugbypy.match import *
from rugbypy.competition import *
from rugbypy.team import *
from rugbypy.venue import *
from rugbypy.player import *

In [111]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [112]:
## Used to eliminate dumb warnings

pd.options.mode.chained_assignment = None

In [113]:
## Reading in data

test_matches = pd.read_excel("detailed_test_matches.xlsx") ## Description of match contexts and the final score of each match
match_stats = pd.read_excel("aggregated_stats.xlsx") ## Match stats collected using player data instead of team data
test_nations = pd.read_excel("test_nations.xlsx") ## List of all Tier 1 Test Nations
team_stats = pd.read_excel("team_stats.xlsx") ## Match stats by team
venues = fetch_all_venues() ## All venues in the database

Fetching all venues...


In [114]:
## Only including test matches between Tier 1 Nations

test_matches = test_matches[test_matches['home_team'].isin(test_nations['team_name']) &
                            test_matches['away_team'].isin(test_nations['team_name'])]

In [115]:
## Removal of Fiji - no national stadium where they're the home team

test_matches = test_matches[test_matches['home_team'] != "Fiji"]

In [116]:
## Data Cleaning

## Only selecting the necessary variabled for outcome
test_match_results = test_matches[['match_id', 'date', 'season', 'competition_id', 'competition', 'venue_id', 'venue', 
                                    'home_team', 'home_team_id', 'away_team', 'away_team_id', 'home_score', 'away_score']]

## Creating a winner variable
test_match_results['winner'] = np.where(test_match_results['home_score'] == test_match_results['away_score'],
                                "Draw", np.where(test_match_results['home_score'] > test_match_results['away_score'], 
                                test_match_results['home_team'], test_match_results['away_team']))

## Creating a new variable classify if the home team won or not
test_match_results['home_team_win'] = np.where(test_match_results['winner'] == test_match_results['home_team'], 1, 0)

## Designating all RWC matches as neutral matches outside of the matches that featured France (hosted the tournament)
test_match_results['neutral_match'] = np.where((test_match_results['competition'] == "Rugby World Cup") &
                                                (test_match_results['home_team'] != "France"),
                                                "neutral", "NA")

## test_match_results - information on all test matches including tests where there is no clear home team (neutral match)

home_away_tests = test_match_results[test_match_results['neutral_match'] != "neutral"]

## home_away_tests

## home_away_tests - test matches where there is a clear home/away distinction. Ignores neutral-site 


In [117]:
## Removal of New Zealand v. South Africa at Twickenham (neutral-site RWC Preparation Match)

home_away_tests = home_away_tests[~home_away_tests['match_id'].isin(['db7135be'])]

In [118]:
## Only included venues that have hosted a test match

test_venues = venues[venues['venue_id'].isin(test_match_results['venue_id'])]

## Validation of the venues list; removed Wembley Stadium and Soldier Field

test_venues = test_venues[~test_venues['venue_name'].isin(['Wembley Stadium', 'Soldier Field'])] 

In [119]:
## Only included match stats for Test 1 Teams

match_stats = match_stats[match_stats['team'].isin(test_nations['team_name'])]

In [120]:
## Data validation

match_stats = match_stats.rename(columns = {"team": "team_name"})

match_stats = match_stats[match_stats['turnovers_lost'] < 30] ## Some matches had teams who committed 100+ turnovers 


In [121]:
## Merging test nations with their associated match stats

match_stats = pd.merge(test_nations, match_stats, how = 'inner', on = 'team_name')

## match_stats


In [122]:
## Joining data columns of stats not collected in the dataset using player data

team_stats = team_stats[['team_id', 'team', 'game_date', 'team_vs', 'team_vs_id', 'match_id', '22m_entries', '22m_conversion',
                        'post_contact_metres', 'ruck_speed_0_3_pct', 'ruck_speed_3_6_pct', 'ruck_speed_6_plus_pct',
                        'possession', 'territory', 'scrums_total', 'mauls_total', 'rucks_total']] 
## All stats not collected using player data


team_stats = team_stats.rename(columns={"team": "team_name"}) # renamed for the purpose of joining

In [123]:
## Join this with data collected from players data set

potential_model_set = pd.merge(team_stats, match_stats, how="left", on = ['team_id', 'team_name', 'match_id'])

In [124]:
## Home/Away Designation

## Intended columns to drop; all associated with each team and associated score. game_date is a redundant column. 

drop_cols = ['team_name', 'team_vs', 'team_vs_id', 'game_date', 'score', 'team_vs_score']

potential_model_set.drop(columns = [c for c in drop_cols if c in potential_model_set.columns])

stat_cols = [c for c in potential_model_set.columns if c not in ('team_id', 'match_id')]

## Creates a new data frame that adds the home designation for the statistics associated with the home team

home_team_stats = potential_model_set.rename(columns={
    'team_id': 'home_team_id',
    **{c: f'home_{c}' for c in stat_cols}
})

## Creates a new data frame that adds the away designation for the statistics associated with the away team

away_team_stats = potential_model_set.rename(columns={
    'team_id': 'away_team_id',
    **{c: f'away_{c}' for c in stat_cols}
})

## Joins both data sets

potential_model_set = pd.merge(test_match_results, home_team_stats, on = ['match_id', 'home_team_id'], how = 'left')

potential_model_set = potential_model_set.merge(away_team_stats, on = ['match_id', 'away_team_id'], how = 'left')

In [125]:
## Removing Neutral Matches

potential_model_set = potential_model_set[potential_model_set['neutral_match'] != "neutral"]

potential_model_set = potential_model_set.drop(columns=['neutral_match', 'home_game_date', 'away_game_date'])



In [126]:
## Adding potential features to model

model_set = potential_model_set[['match_id', 'date', 'season', 'competition_id', 'competition','venue_id', 'venue', 
                                        'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_score', 'away_score',
                                        'winner', 'home_team_win', 'home_carries', 'away_carries', 'home_line_breaks',
                                        'away_line_breaks', 'home_tackles_made', 'away_tackles_made', 'home_tackles_missed',
                                        'away_tackles_missed', 'home_kicks', 'away_kicks', 'home_metres_carried',
                                        'away_metres_carried', 'home_defenders_beaten', 'away_defenders_beaten',
                                        'home_yellow_cards', 'away_yellow_cards', 
                                        'home_red_cards', 'away_red_cards', 'home_tries', 'away_tries', 'home_penalties_conceded', 'away_penalties_conceded', 
                                         'home_penalty_goals', 'away_penalty_goals']]

In [127]:
model_set = model_set.dropna()

model_set

,match_id,date,season,competition_id,competition,venue_id,venue,home_team_id,home_team,away_team_id,...,home_yellow_cards,away_yellow_cards,home_red_cards,away_red_cards,home_tries,away_tries,home_penalties_conceded,away_penalties_conceded,home_penalty_goals,away_penalty_goals
29,99ea4482,20220806,2022,423a2845,The Rugby Championship,f2408621,Estadio Malvinas Argentinas,11392c0c,Argentina,b904fef9,...,1.0,0.0,0.0,0.0,2.0,4.0,11.0,16.0,4.0,2.0
30,860ce2d2,20220806,2022,423a2845,The Rugby Championship,879c72e0,Mbombela Stadium,9b93d911,South Africa,d75e45cc,...,0.0,0.0,1.0,0.0,2.0,1.0,7.0,12.0,3.0,1.0
31,9233d4bc,20220813,2022,423a2845,The Rugby Championship,e952f544,Ellis Park,9b93d911,South Africa,d75e45cc,...,1.0,1.0,0.0,0.0,2.0,4.0,7.0,12.0,3.0,3.0
32,2864ca3c,20220813,2022,423a2845,The Rugby Championship,4f121a27,Estadio Bicentenario,11392c0c,Argentina,b904fef9,...,0.0,1.0,0.0,0.0,7.0,2.0,9.0,13.0,1.0,1.0
33,f4f5d078,20220827,2022,423a2845,The Rugby Championship,6c7f7498,Adelaide Oval,b904fef9,Australia,9b93d911,...,2.0,1.0,0.0,0.0,3.0,2.0,16.0,9.0,2.0,1.0
34,d661c5a6,20220827,2022,423a2845,The Rugby Championship,0cf3fe1f,Apollo Projects Stadium,d75e45cc,New Zealand,11392c0c,...,1.0,0.0,0.0,0.0,2.0,1.0,14.0,11.0,2.0,6.0
35,d8466f79,20220903,2022,423a2845,The Rugby Championship,bad3bef0,Allianz Stadium,b904fef9,Australia,9b93d911,...,1.0,2.0,0.0,0.0,1.0,4.0,13.0,13.0,1.0,0.0
36,d2b35ab2,20220903,2022,423a2845,The Rugby Championship,36e3ebc0,FMG Stadium Waikato,d75e45cc,New Zealand,11392c0c,...,1.0,1.0,0.0,0.0,7.0,0.0,8.0,11.0,2.0,1.0
37,c51fac62,20220915,2022,423a2845,The Rugby Championship,5b9e1472,Marvel Stadium,b904fef9,Australia,d75e45cc,...,3.0,1.0,0.0,0.0,4.0,5.0,10.0,12.0,3.0,2.0
38,0c15dfc8,20220917,2022,423a2845,The Rugby Championship,83c02fe2,Estadio Libertadores de América,11392c0c,Argentina,9b93d911,...,2.0,2.0,0.0,0.0,1.0,4.0,17.0,16.0,2.0,1.0


In [60]:
## Elo Rating Development

def expected_score(rating_a, rating_b):
    return 1/(1 + 10 **((rating_b - rating_a)/400))

def update_elo(rating, expected, actual, k = 20):
    return rating + k * (actual - expected)

def run_elo(matches, win_pct_seeds, initial_rating = 1000, k = 20):
    matches = matches.sort_values('date').reset_index(drop = True)
    seed_lookup = win_pct_seeds.set_index(['team_id', 'seed_season'])['elo_seed'].to_dict()
    ratings = {}
    records = []
    current_season = None

    for x, row in matches.iterrows():
        home_id = row['home_team_id']
        away_id = row['away_team_id']
        season = row['season']

        if season != current_season:
            current_season = season
        for id in ratings:
            seed = seed_lookup.get((id, season), ratings[id])
            ratings[id] = seed
    
        ratings.setdefault(home_id, seed_lookup.get((home_id, season), initial_rating))
        ratings.setdefault(away_id, seed_lookup.get((away_id, season), initial_rating))

        r_home = ratings[home_id]
        r_away = ratings[away_id]

        ratings.setdefault(home_id, )

        p_home = expected_score(r_home, r_away)

        if row['home_score'] > row['away_score']:
            outcome = 1
        elif row['home_score'] == row['away_score']:
            outcome = 0.5
        else:
            outcome = 0
        
        records.append({
            'match_id': row['match_id'],
            'date': row['date'],
            'home_team_id': home_id,
            'away_team_id': away_id,
            'pre_elo_home': r_home,
            'pre_elo_away': r_away,
            'elo_diff': r_home - r_away,
            'predicted_prob': p_home,
            'actual_outcome': outcome,
            'wpa': outcome - p_home
        })

        ratings[home_id] = update_elo(r_home, p_home, outcome, k)
        ratings[away_id] = update_elo(r_away, 1 - p_home, 1 - outcome, k)

    results_df = pd.DataFrame(records)
    final_ratings = pd.Series(ratings, name = 'elo')

    return results_df, final_ratings

def elo_from_win_pct(win_pct, base=1000, scale=200):
    return base + scale * (win_pct - 0.5)



In [61]:
## Adding season form to elo rating

test_matches['winner'] = np.where(test_matches['home_score'] == test_matches['away_score'],
                                "Draw", np.where(test_matches['home_score'] > test_matches['away_score'], 
                                test_matches['home_team'], test_matches['away_team']))

home_matches = test_matches[['season', 'home_team_id', 'home_team', 'home_score', 'away_score', 'winner']].copy()
home_matches['team_id'] = home_matches['home_team_id']
home_matches['team'] = home_matches['home_team']
home_matches['win'] = np.where(home_matches['home_team'] == home_matches['winner'], 1, 0)

away_matches = test_matches[['season', 'away_team_id', 'away_team', 'home_score', 'away_score', 'winner']].copy()
away_matches['team_id'] = away_matches['away_team_id']
away_matches['team'] = away_matches['away_team']
away_matches['win'] = np.where(away_matches['away_team'] == away_matches['winner'], 1, 0)

team_matches = pd.concat([
    home_matches[['season', 'team_id', 'team', 'win']],
    away_matches[['season', 'team_id', 'team', 'win']]
], ignore_index = True)

season_summary = (
    team_matches
    .groupby(['season', 'team_id', 'team'])
    .agg(
        games_played = ('team', 'size'),
        wins = ('win', 'sum')
    )
    .reset_index()
)

season_summary['win_pct'] = np.round((season_summary['wins']/season_summary['games_played']), 2)

season_summary['seed_season'] = season_summary['season'] + 1
season_summary['elo_seed'] = season_summary['win_pct'].apply(elo_from_win_pct)

In [62]:
## Running the elo rating

results, final_ratings = run_elo(test_matches, season_summary, k = 20)

In [63]:
## Joining elo ratings with matches in the model set

model_set = model_set.merge(results, how = 'inner', on = ['match_id', 'date', 'home_team_id', 'away_team_id'])




In [64]:
## Percentages

model_set['home_tackle_success_rate'] = np.round((model_set['home_tackles_made']/(model_set['home_tackles_missed'] + model_set['home_tackles_made'])) * 100, 2)
model_set['away_tackle_success_rate'] = np.round((model_set['away_tackles_made']/(model_set['away_tackles_made'] + model_set['away_tackles_missed'])) * 100, 2)


In [65]:
## Data Validation - Erroneous data

garbage = ['b8980cf7',  'd661c5a6', 'de4008a7', '99ea4482', '1e174fdb']

model_set = model_set[~model_set['match_id'].isin(garbage)]



In [66]:
## Using differences rather than raw numbers for both sides

skip_cols = ['home_team', 'home_team_id', 'away_team', 'away_team_id', 'home_score',
            'away_score', 'home_team_win']


home_cols = [c for c in model_set.columns if c.startswith('home_') and c not in skip_cols]
away_cols = [c for c in model_set.columns if c.startswith('away_') and c not in skip_cols]

for col in home_cols + away_cols:
    model_set[col] = pd.to_numeric(model_set[col], errors='coerce')

for home_col in home_cols:
    stat = home_col.replace('home_', '')
    away_col = f'away_{stat}'
    if away_col in model_set.columns:
        model_set[f'{stat}_diff'] = model_set[home_col] - model_set[away_col]


diff_cols = [c for c in model_set.columns if c.endswith('_diff')]
print(model_set[diff_cols].head())
print(model_set[diff_cols].notna().sum())



   elo_diff  carries_diff  line_breaks_diff  tackles_made_diff  \
0       0.0          53.0               5.0              -67.0   
1       0.0         -21.0               2.0               29.0   
2       0.0          44.0               5.0              -58.0   
3     -20.0          -7.0              -4.0               15.0   
4       0.0         -11.0               0.0              -15.0   

   tackles_missed_diff  kicks_diff  metres_carried_diff  \
0                -12.0         9.0                301.0   
1                 -9.0        -5.0               -117.0   
2                -16.0         3.0                227.0   
3                 12.0         9.0                 -4.0   
4                 -5.0         0.0                -22.0   

   defenders_beaten_diff  yellow_cards_diff  red_cards_diff  tries_diff  \
0                   12.0               -1.0             0.0         3.0   
1                    9.0               -1.0             0.0         0.0   
2                   16.

In [67]:
## Condensing the Model Set - need continued validation :(


model_set = model_set[['match_id', 'date', 'season', 'competition_id', 'competition','venue_id', 'venue', 
                        'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_score', 'away_score',
                        'winner', 'home_team_win', 'pre_elo_home', 'pre_elo_away', 'elo_diff', 'carries_diff',
                        'line_breaks_diff', 'tackle_success_rate_diff', 'kicks_diff', 'metres_carried_diff',
                        'defenders_beaten_diff', 'yellow_cards_diff', 'red_cards_diff', 'penalties_conceded_diff',
                        'penalty_goals_diff']]
